# DistressNet: this week's experiments (calm enrollment, seeds, ASR, safety, latency)

Runs every experiment from the P0/P1/P2 list that needs the audio, the GPU or a language model, from the branch `claude/project-thread-9dotrd` of github.com/Paresh140/distress-detection. No WavLM training: the encoder stays frozen and every clip is embedded once and cached, then only small heads train.

**Settings (right-hand panel, before running):**
1. **Session options → Accelerator → GPU T4 x2** (or P100).
2. **Session options → Internet → On** (GitHub, pip, WavLM and Whisper weights, Ollama).
3. Datasets attach themselves (`uwrfkaggler/ravdess-emotional-speech-audio`, `ejlok1/cremad`).

**Run:** **Save Version → Save & Run All**. The first pass embeds about 5,800 clips (roughly 15 to 25 minutes on a T4). The calm-enrollment study trains about 36 small heads, so plan on 1.5 to 2.5 hours for the whole notebook; a Kaggle session allows 12. When it finishes, download `distressnet_week_results.zip` from the Output tab and choose **Load Kaggle results** on the studio's Training page (it installs the new `voice_head.pt` with the calm-enrolled head, and the reports). Or send the zip back so the numbers go into the paper.

What each step produces:

| step | script | answers |
|---|---|---|
| 1 | `app.voice` | live head + calm-enrolled head (P0.1) |
| 2 | `app.enroll_eval` | calm enrollment vs none vs full speaker norm, per held-out speaker (P0.2) |
| 3 | `app.seed_eval` | seed variance of the RAVDESS headline (P1.8) |
| 4 | `app.cv --head four-mean / binary-attentive / binary-mean` | pooling and label ablation (P2.12), per-clip probabilities |
| 5 | `app.ece_eval` | ECE at 5/10/15 bins + reliability diagram (P2.13) |
| 6 | `app.asr_eval` | Whisper base WER, calm vs distressed (P1.6) |
| 7 | Ollama + `app.reader_eval`, `app.safety_eval`, `app.safety_split --model`, `app.pipeline_eval` | LLM reader (P0.3), held-out safety with the model (P0.5), mismatch and joint false alarms with the LLM (P0.4, P1.7) |
| 8 | `app.latency_eval` | per-stage latency, GPU and CPU (P2.14) |


In [ ]:
import os
import subprocess
from pathlib import Path

REPO = "https://github.com/Paresh140/distress-detection.git"
BRANCH = "claude/project-thread-9dotrd"
REPO_DIR = "/kaggle/working/distress-detection"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "-q", "origin", BRANCH], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], check=True)
subprocess.run(["pip", "-q", "install", "opensmile", "faster-whisper"], check=True)

import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none. Turn on a GPU under Session options, then run again.")
BACKEND = Path(REPO_DIR) / "backend"
ARTIFACTS = BACKEND / "artifacts"


def run(*args: str, env: dict | None = None, check: bool = True) -> None:
    """Run one project script from backend/ (output streams as it runs)."""
    print("\n$ python -m", " ".join(args), flush=True)
    subprocess.run(["python", "-m", *args], cwd=BACKEND, check=check, env={**os.environ, **(env or {})})


In [ ]:
import glob
import shutil


def attach(handle: str, pattern: str) -> list[str]:
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        import kagglehub

        root = kagglehub.dataset_download(handle)
        hits = sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))
    assert hits, f"Could not find {handle}. Add it with Add Input, then run this cell again."
    return hits


ravdess = os.path.dirname(attach("uwrfkaggler/ravdess-emotional-speech-audio", "audio_speech_actors_01-24/Actor_01")[0])
crema = os.path.dirname(attach("ejlok1/cremad", "1001_DFA_ANG_XX.wav")[0])
data = Path(REPO_DIR) / "data"
data.mkdir(exist_ok=True)
for name, target in (("ravdess", ravdess), ("crema-d", crema)):
    link = data / name
    if link.is_symlink() or link.is_file():
        link.unlink()
    elif link.is_dir():
        shutil.rmtree(link)
    link.symlink_to(target)
print("RAVDESS:", len(glob.glob(f"{data}/ravdess/Actor_*/03-01-*.wav")), "clips;  CREMA-D:", len(glob.glob(f"{data}/crema-d/*.wav")), "clips")


## 1. Live voice model with the calm-enrolled head (embeds every clip once)

In [ ]:
run("app.voice")


## 2. Calm enrollment on held-out speakers

In [ ]:
run("app.enroll_eval", "--draws", "3", "--sweep", "1", "3", "10")


## 3. Seed variance of the RAVDESS headline

In [ ]:
run("app.seed_eval", "--seeds", "7", "17", "27")


## 4. Pooling and label ablation (the heads that already exist), and per-clip probabilities for ECE

In [ ]:
for head in ("four-mean", "binary-attentive", "binary-mean"):
    run("app.cv", "--only", "wavlm", "--head", head)


## 5. ECE sensitivity and reliability diagram

In [ ]:
run("app.ece_eval")
from IPython.display import Image, display

display(Image(str(ARTIFACTS / "reliability.png")))


## 6. Whisper base word error rate on acted speech

In [ ]:
run("app.asr_eval", "--per-class", "25")


## 7. Language model steps with a local Ollama (llama3.2:3b on the GPU)

This installs Ollama in the notebook, the same local model the studio falls back to. If you would rather use gpt-4o-mini, skip the first cell and set `os.environ["OPENAI_API_KEY"]` from a Kaggle secret instead (never paste the key into the notebook). If Ollama will not install, the rest of the notebook still runs; these steps then fall back to the phrase list and say so.

In [ ]:
import time
import urllib.request

subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=False)
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)
subprocess.run(["ollama", "pull", "llama3.2:3b"], check=False)
os.environ["OLLAMA_MODEL"] = "llama3.2:3b"


In [ ]:
run("app.reader_eval", "--readers", "llm", check=False)
run("app.safety_eval", check=False)            # exits 1 when a target is missed; the report is still written
run("app.safety_split", "--model", check=False)
run("app.pipeline_eval", "--readers", "phrases", "roberta", "llm", check=False)


## 8. Latency of one reading (GPU, then CPU only)

In [ ]:
clip = sorted(glob.glob(f"{data}/ravdess/Actor_21/03-01-04-*.wav"))[0]
run("app.latency_eval", "--clip", clip, "--runs", "10")
run("app.latency_eval", "--clip", clip, "--runs", "5", "--out", "latency_report_cpu.json", env={"CUDA_VISIBLE_DEVICES": ""})


## Package the results

In [ ]:
OUTPUT = [
    "voice_head.pt", "voice_report.json", "enrollment_report.json", "seed_report.json", "cv_report.json",
    "ece_report.json", "reliability.png", "asr_report.json", "reader_report.json", "safety_report.json",
    "safety_heldout_report.json", "pipeline_report.json", "latency_report.json", "latency_report_cpu.json",
]
out_dir = Path("/kaggle/working/distressnet_week_results")
shutil.rmtree(out_dir, ignore_errors=True)
out_dir.mkdir()
for name in OUTPUT:
    if (ARTIFACTS / name).exists():
        shutil.copy2(ARTIFACTS / name, out_dir / name)
    else:
        print("missing:", name)
shutil.make_archive("/kaggle/working/distressnet_week_results", "zip", out_dir)
print(sorted(p.name for p in out_dir.iterdir()))
import json

if (ARTIFACTS / "enrollment_report.json").exists():
    print(json.loads((ARTIFACTS / "enrollment_report.json").read_text())["table"])

from IPython.display import FileLink

FileLink("distressnet_week_results.zip")
